### Imports

In [1]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

### Load Data

In [2]:
# Step 1: extract column names
with open("conn.log.labeled", "r") as f:
    for line in f:
        if line.startswith("#fields"):
            columns = line.strip().split("\t")[1:]
            break

# Step 2: load data correctly
df = pd.read_csv(
    "conn.log.labeled",
    sep="\t",
    comment="#",
    names=columns,
    low_memory=False
)
df = pd.DataFrame(df)

pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.width', 1000)         # set width wide enough
pd.set_option('display.max_colwidth', 100)   # increase column width

print(df.tail(15))

                 ts                 uid        id.orig_h  id.orig_p        id.resp_h  id.resp_p proto service  duration orig_bytes resp_bytes conn_state local_orig local_resp  missed_bytes history  orig_pkts  orig_ip_bytes  resp_pkts  resp_ip_bytes tunnel_parents   label   detailed-label
10388  1.533129e+09  CYvKN52inQUxQxKMq3  192.168.100.113      38248   128.185.250.50         50   tcp       -         -          -          -         S0          -          -             0       S          1             60          0              0               (empty)   Malicious   C&C
10389  1.533129e+09  Ctl4lz34icvc1W3gaj  192.168.100.113        123   37.157.198.150        123   udp       -  0.004995         48         48         SF          -          -             0      Dd          1             76          1             76                    (empty)   Benign   -
10390  1.533129e+09   CYAbwqwjNKhBYuJs4  192.168.100.113      38248   128.185.250.50         50   tcp       -         -          -   

### Data Wrangling

In [3]:
broken_col = "tunnel_parents   label   detailed-label"

# initialize new columns
df["tunnel_parents"] = "(empty)"
df["label"] = "Benign"
df["detailed_label"] = "-"

for idx, val in df[broken_col].items():
    if isinstance(val, str) and "Malicious" in val:
        df.at[idx, "label"] = "Malicious"
        df.at[idx, "detailed_label"] = val.split("Malicious", 1)[1].strip()

df.drop(columns=[broken_col], inplace=True)
df.drop(columns=["uid"], inplace=True)  # drop uid column

In [4]:
df

,ts,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,resp_bytes,conn_state,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,label,detailed_label
0,1.533043e+09,192.168.100.113,123,81.2.254.224,123,udp,-,0.005490,48,48,SF,-,-,0,Dd,1,76,1,76,(empty),Benign,-
1,1.533043e+09,192.168.100.113,123,147.231.100.5,123,udp,-,0.001741,48,48,SF,-,-,0,Dd,1,76,1,76,(empty),Benign,-
2,1.533043e+09,192.168.100.113,123,31.31.74.35,123,udp,-,0.004495,48,48,SF,-,-,0,Dd,1,76,1,76,(empty),Benign,-
3,1.533043e+09,192.168.100.113,123,147.251.48.140,123,udp,-,0.006988,48,48,SF,-,-,0,Dd,1,76,1,76,(empty),Benign,-
4,1.533043e+09,192.168.100.113,123,147.231.100.5,123,udp,-,0.001487,48,48,SF,-,-,0,Dd,1,76,1,76,(empty),Benign,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10398,1.533129e+09,192.168.100.113,47432,178.128.185.250,50,tcp,-,-,-,-,S0,-,-,0,S,1,60,0,0,(empty),Malicious,C&C
10399,1.533129e+09,192.168.100.113,38252,128.185.250.50,50,tcp,-,3.089221,0,0,S0,-,-,0,S,3,180,0,0,(empty),Malicious,C&C
10400,1.533129e+09,192.168.100.113,38252,128.185.250.50,50,tcp,-,-,-,-,S0,-,-,0,S,1,60,0,0,(empty),Malicious,C&C
10401,1.533129e+09,192.168.100.113,123,147.231.100.5,123,udp,-,0.001988,48,48,SF,-,-,0,Dd,1,76,1,76,(empty),Benign,-


In [5]:
# I want to check the number of unique values in each column
for col in df.columns:
    unique_values = df[col].nunique()
    if unique_values < 2:
        df.drop(columns=[col], inplace=True)
    print(f"Column: {col}, Unique Values: {unique_values}")



Column: ts, Unique Values: 10403
Column: id.orig_h, Unique Values: 2
Column: id.orig_p, Unique Values: 2059
Column: id.resp_h, Unique Values: 9
Column: id.resp_p, Unique Values: 3
Column: proto, Unique Values: 2
Column: service, Unique Values: 1
Column: duration, Unique Values: 2056
Column: orig_bytes, Unique Values: 3
Column: resp_bytes, Unique Values: 3
Column: conn_state, Unique Values: 3
Column: local_orig, Unique Values: 1
Column: local_resp, Unique Values: 1
Column: missed_bytes, Unique Values: 1
Column: history, Unique Values: 4
Column: orig_pkts, Unique Values: 3
Column: orig_ip_bytes, Unique Values: 4
Column: resp_pkts, Unique Values: 2
Column: resp_ip_bytes, Unique Values: 3
Column: tunnel_parents, Unique Values: 1
Column: label, Unique Values: 2
Column: detailed_label, Unique Values: 2


In [6]:
df

,ts,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,duration,orig_bytes,resp_bytes,conn_state,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,label,detailed_label
0,1.533043e+09,192.168.100.113,123,81.2.254.224,123,udp,0.005490,48,48,SF,Dd,1,76,1,76,Benign,-
1,1.533043e+09,192.168.100.113,123,147.231.100.5,123,udp,0.001741,48,48,SF,Dd,1,76,1,76,Benign,-
2,1.533043e+09,192.168.100.113,123,31.31.74.35,123,udp,0.004495,48,48,SF,Dd,1,76,1,76,Benign,-
3,1.533043e+09,192.168.100.113,123,147.251.48.140,123,udp,0.006988,48,48,SF,Dd,1,76,1,76,Benign,-
4,1.533043e+09,192.168.100.113,123,147.231.100.5,123,udp,0.001487,48,48,SF,Dd,1,76,1,76,Benign,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10398,1.533129e+09,192.168.100.113,47432,178.128.185.250,50,tcp,-,-,-,S0,S,1,60,0,0,Malicious,C&C
10399,1.533129e+09,192.168.100.113,38252,128.185.250.50,50,tcp,3.089221,0,0,S0,S,3,180,0,0,Malicious,C&C
10400,1.533129e+09,192.168.100.113,38252,128.185.250.50,50,tcp,-,-,-,S0,S,1,60,0,0,Malicious,C&C
10401,1.533129e+09,192.168.100.113,123,147.231.100.5,123,udp,0.001988,48,48,SF,Dd,1,76,1,76,Benign,-


In [7]:
#Label Encoding
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])
df["detailed_label_encoded"] = le.fit_transform(df["detailed_label"])

# One Hot Encoding
ohe = OneHotEncoder(sparse_output=False)
columns_to_encode = ["id.orig_h", "id.resp_h", "id.resp_p", "proto", "orig_bytes", "resp_bytes", "conn_state", "history", "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes"]
encoded_features = ohe.fit_transform(df[columns_to_encode])
df = pd.concat([df, pd.DataFrame(encoded_features, columns=ohe.get_feature_names_out(columns_to_encode))], axis=1)

# df.drop(columns=columns_to_encode, inplace=True)

In [8]:
df

,ts,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,duration,orig_bytes,resp_bytes,conn_state,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,label,detailed_label,label_encoded,detailed_label_encoded,id.orig_h_192.168.100.102,id.orig_h_192.168.100.113,id.resp_h_128.185.250.50,id.resp_h_147.231.100.5,id.resp_h_147.251.48.140,id.resp_h_178.128.185.250,id.resp_h_192.168.100.113,id.resp_h_31.31.74.35,id.resp_h_37.157.198.150,id.resp_h_80.79.25.111,id.resp_h_81.2.254.224,id.resp_p_22,id.resp_p_50,id.resp_p_123,proto_tcp,proto_udp,orig_bytes_-,orig_bytes_0,orig_bytes_48,resp_bytes_-,resp_bytes_0,resp_bytes_48,conn_state_OTH,conn_state_S0,conn_state_SF,history_D,history_Dd,history_S,history_^d,orig_pkts_0,orig_pkts_1,orig_pkts_3,orig_ip_bytes_0,orig_ip_bytes_60,orig_ip_bytes_76,orig_ip_bytes_180,resp_pkts_0,resp_pkts_1,resp_ip_bytes_0,resp_ip_bytes_76,resp_ip_bytes_88
0,1.533043e+09,192.168.100.113,123,81.2.254.224,123,udp,0.005490,48,48,SF,Dd,1,76,1,76,Benign,-,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1,1.533043e+09,192.168.100.113,123,147.231.100.5,123,udp,0.001741,48,48,SF,Dd,1,76,1,76,Benign,-,0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
2,1.533043e+09,192.168.100.113,123,31.31.74.35,123,udp,0.004495,48,48,SF,Dd,1,76,1,76,Benign,-,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,1.533043e+09,192.168.100.113,123,147.251.48.140,123,udp,0.006988,48,48,SF,Dd,1,76,1,76,Benign,-,0,0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
4,1.533043e+09,192.168.100.113,123,147.231.100.5,123,udp,0.001487,48,48,SF,Dd,1,76,1,76,Benign,-,0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10398,1.533129e+09,192.168.100.113,47432,178.128.185.250,50,tcp,-,-,-,S0,S,1,60,0,0,Malicious,C&C,1,1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
10399,1.533129e+09,192.168.100.113,38252,128.185.250.50,50,tcp,3.089221,0,0,S0,S,3,180,0,0,Malicious,C&C,1,1,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
10400,1.533129e+09,192.168.100.113,38252,128.185.250.50,50,tcp,-,-,-,S0,S,1,60,0,0,Malicious,C&C,1,1,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
10401,1.533129e+09,192.168.100.113,123,147.231.100.5,123,udp,0.001988,48,48,SF,Dd,1,76,1,76,Benign,-,0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


### Feature Selection

In [9]:
# Feature Selection Using Variance Threshold
# Drop the original categorical columns before applying VarianceThreshold
cols_to_drop = columns_to_encode + ["label", "detailed_label", "ts", "id.orig_p"]
df.drop(columns=cols_to_drop, inplace=True)

# Replace "-" with 0 and convert to numeric
df['duration'] = df['duration'].replace('-', 0).astype(float)

In [10]:
vt = VarianceThreshold(0.0)

X = df.copy()  # exclude targets
X.drop(columns=["label_encoded", "detailed_label_encoded"], inplace=True)
X = X.loc[:, vt.fit(X).variances_ > 0]

In [11]:
# Add back the target columns
X["label_encoded"] = df["label_encoded"]
X["detailed_label_encoded"] = df["detailed_label_encoded"]

X

,duration,id.orig_h_192.168.100.102,id.orig_h_192.168.100.113,id.resp_h_128.185.250.50,id.resp_h_147.231.100.5,id.resp_h_147.251.48.140,id.resp_h_178.128.185.250,id.resp_h_192.168.100.113,id.resp_h_31.31.74.35,id.resp_h_37.157.198.150,id.resp_h_80.79.25.111,id.resp_h_81.2.254.224,id.resp_p_22,id.resp_p_50,id.resp_p_123,proto_tcp,proto_udp,orig_bytes_-,orig_bytes_0,orig_bytes_48,resp_bytes_-,resp_bytes_0,resp_bytes_48,conn_state_OTH,conn_state_S0,conn_state_SF,history_D,history_Dd,history_S,history_^d,orig_pkts_0,orig_pkts_1,orig_pkts_3,orig_ip_bytes_0,orig_ip_bytes_60,orig_ip_bytes_76,orig_ip_bytes_180,resp_pkts_0,resp_pkts_1,resp_ip_bytes_0,resp_ip_bytes_76,resp_ip_bytes_88,label_encoded,detailed_label_encoded
0,0.005490,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0
1,0.001741,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0
2,0.004495,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0
3,0.006988,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0
4,0.001487,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10398,0.000000,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1,1
10399,3.089221,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1,1
10400,0.000000,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1,1
10401,0.001988,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0,0


In [12]:
for value in X["label_encoded"].unique():
    print(f"Label: {value}, Count: {len(X[X['label_encoded'] == value])}")

Label: 0, Count: 2181
Label: 1, Count: 8222


In [13]:
# Feature Selection Using Feature Importance from Random Forest
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X.drop(columns=["label_encoded", "detailed_label_encoded"]), X["label_encoded"])
importances = rf.feature_importances_
feature_names = X.drop(columns=["label_encoded", "detailed_label_encoded"]).columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
print(feature_importance_df.sort_values(by='Importance', ascending=False).head(20))  # Display top 20 important features

                    Feature  Importance
28                history_S    0.140523
35         orig_ip_bytes_76    0.140224
14            id.resp_p_123    0.100227
13             id.resp_p_50    0.091173
16                proto_udp    0.090337
15                proto_tcp    0.084600
24            conn_state_S0    0.059460
22            resp_bytes_48    0.045571
37              resp_pkts_0    0.039614
39          resp_ip_bytes_0    0.039578
38              resp_pkts_1    0.029677
27               history_Dd    0.029651
40         resp_ip_bytes_76    0.025803
19            orig_bytes_48    0.019804
25            conn_state_SF    0.019748
0                  duration    0.009993
34         orig_ip_bytes_60    0.007672
32              orig_pkts_3    0.006094
18             orig_bytes_0    0.006087
4   id.resp_h_147.231.100.5    0.005353


In [14]:
# Select top 20 features based on importance
top_features = feature_importance_df.head(20)['Feature'].tolist()
X = X[top_features + ["label_encoded", "detailed_label_encoded"]]

X

,history_S,orig_ip_bytes_76,id.resp_p_123,id.resp_p_50,proto_udp,proto_tcp,conn_state_S0,resp_bytes_48,resp_pkts_0,resp_ip_bytes_0,resp_pkts_1,history_Dd,resp_ip_bytes_76,orig_bytes_48,conn_state_SF,duration,orig_ip_bytes_60,orig_pkts_3,orig_bytes_0,id.resp_h_147.231.100.5,label_encoded,detailed_label_encoded
0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.005490,0.0,0.0,0.0,0.0,0,0
1,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.001741,0.0,0.0,0.0,1.0,0,0
2,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.004495,0.0,0.0,0.0,0.0,0,0
3,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.006988,0.0,0.0,0.0,0.0,0,0
4,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.001487,0.0,0.0,0.0,1.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10398,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.0,0.0,0.0,0.0,1,1
10399,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,3.089221,0.0,1.0,1.0,0.0,1,1
10400,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.000000,1.0,0.0,0.0,0.0,1,1
10401,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.001988,0.0,0.0,0.0,1.0,0,0


In [15]:
X.to_csv("processed_iot_data.csv", index=False)

### Model Training

In [16]:
# Train a Random Forest Classifier

X = X.drop(columns=["detailed_label_encoded", "label_encoded"])
y = df["label_encoded"]  # For binary classification, drop detailed label

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(
    n_estimators=100,       # fewer trees
    max_depth=5,            # limit tree depth
    min_samples_split=10,   # require more samples to split
    min_samples_leaf=5,     # require more samples at leaf
    max_features='sqrt',    # only consider sqrt(features) at each split
    random_state=42,
    class_weight='balanced'
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[ 446    0]
 [   0 1635]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       446
           1       1.00      1.00      1.00      1635

    accuracy                           1.00      2081
   macro avg       1.00      1.00      1.00      2081
weighted avg       1.00      1.00      1.00      2081



In [17]:
# NOTE:
# The Random Forest model achieves 1.0 accuracy, precision, recall, and F1-score on this dataset.
# This is due to the combination of:
# 1. Highly distinctive features between benign and malicious traffic in this file.
# 2. Only a single conn.log.labeled file was used, making the dataset "clean" and easily separable.
# WARNING: This does NOT guarantee generalization to other IoT devices or files.
# For real-world deployment, more data from multiple captures and devices is required.